### <b>Fine-Tuning Code (Using LoRA) </b>
This script uses PEFT (LoRA) to ensure you can train the model on a modest GPU (like a free T4 in Google Colab).

In [ ]:
# import libraries
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

In [ ]:
# 1. Load your dataset
dataset = load_dataset("Prathamesh25/aptitude-qa-dataset", split="train")

In [ ]:
# 2. Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 3. LoRA Configuration (Efficiency)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Target key layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# 4. Training Arguments
training_args = SFTConfig(
    output_dir="./smollm2-aptitude-agent",
    max_seq_length=512,
    dataset_text_field="messages", # TRL handles the "messages" format automatically
    packing=False,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_steps=100,
    logging_steps=10,
    push_to_hub=False,
)


In [ ]:
# 5. Initialize Trainer
trainer = SFTTrainer(
    model=model_id,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
)

In [ ]:
# 6. Start Fine-Tuning
trainer.train()
trainer.save_model("./smollm2-aptitude-agent")